# RQ IV: Mitigation


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reported_runs/analysis").is_dir())
sys.path.insert(0, str(repo)) if str(repo) not in sys.path else None
from reported_runs.analysis.presentation import load_results, NotebookReport, temporal_table, effect_table, rank_summary, unique_trajectories, label_conditions
from reported_runs.analysis.plotting import trajectory_grid, effect_grid, rank_grid, association_grid
from reported_runs.analysis.comparisons import summarise_iterations
%matplotlib inline

results = load_results()
report = NotebookReport(results, 'rq4')
scales = report.scales
trajectory = label_conditions(results["trajectory"])
sources = results["sources"]
summaries = results["summaries"]
effect_summaries = results["effect_summaries"]
pd.set_option("display.max_columns", 18)
pd.set_option("display.max_rows", 60)

## Availability and matched controls


In [ ]:
rq4_sources = sources[sources['study'].eq('movielens_mitigation_a300_e804')]
report.availability(rq4_sources)
rq4 = trajectory[trajectory['study'].eq('movielens_mitigation_a300_e804')]
rq4_effects = effect_summaries[effect_summaries['study'].eq('movielens_mitigation_a300_e804')]

## Post-processing: total, immediate reranking, and feedback


In [ ]:
from reported_runs.analysis.mitigation_decomposition import load_postprocessing_components
from reported_runs.analysis.mitigation_decomposition_views import DECOMPOSITION_METRICS, COMPONENT_ORDER, build_postprocessing_decomposition, decomposition_scales
post_sources = rq4_sources.loc[rq4_sources.intervention.eq('post_processing')]
post_components = load_postprocessing_components(post_sources, repo, repo / 'reported_runs/analysis/cache/mitigation_decomposition')
controls = rq4.loc[rq4.intervention.eq('control')]
decomposition = build_postprocessing_decomposition(post_components, controls, results['comparison_plan'])
component_effects = decomposition['summaries']
mitigation_scales = decomposition_scales(scales, component_effects)
fig = effect_grid(component_effects, DECOMPOSITION_METRICS, title='Post-processing: total, immediate reranking, and feedback components', scales=mitigation_scales, value='value', facet='window', category='component_label', category_order=COMPONENT_ORDER)
fig._analysis_spec['decomposition'] = 'Total = immediate reranking + feedback/state component; treatment minus reference; full displayed ranks 1–10'
fig._analysis_spec['contrast_description'] = 'Total = immediate reranking + feedback/state component; each component uses its stated reference'
report.figure(fig, component_effects, 'rq4_post_processing_components', "", recommended=False, why="")

In [ ]:
from reported_runs.analysis.plotting import label
for metric in DECOMPOSITION_METRICS:
    report.table(decomposition['tables'][metric], 'rq4_post_processing_' + metric + '_components', "", columns=['model', 'stage', 'seed', 'population_seed', 'window', 'Total (pp)', 'Immediate (pp)', 'Feedback (pp)'], source_frame=post_components)
component_consistency = decomposition['seed_consistency']
for metric in DECOMPOSITION_METRICS:
    selected = component_consistency.loc[component_consistency.metric.eq(metric)]
    report.table(selected, 'rq4_post_processing_' + metric + '_component_seed_consistency', "", columns=['model', 'stage', 'window', 'component_label', 'n_seed_blocks', 'mean_effect', 'min_effect', 'max_effect'], source_frame=post_components)
report.table(decomposition['eligibility'], 'rq4_post_processing_component_eligibility', "", columns=['model', 'stage', 'seed', 'metric', 'window', 'n_post_defined', 'n_original_order_defined', 'n_control_defined', 'n_common_defined', 'identity_defined'], source_frame=post_components)
component_counts = temporal_table(decomposition['count_summaries'], ['canonical_valid_clicks', 'original_order_valid_clicks', 'control_valid_clicks', 'both_arms_valid_clicks'], group_columns=['model', 'stage', 'seed', 'population_seed', 'window'])
report.table(component_counts, 'rq4_post_processing_click_denominators', "", source_frame=post_components)

## Total mitigation effects


In [ ]:
treatment_effects = rq4_effects[rq4_effects['kind'].eq('mitigation')]
fig = effect_grid(treatment_effects, ['head_click_share', 'head_exposure', 'weighted_head_exposure'], title='Total mitigation effect: treatment minus own control', scales=mitigation_scales, value='value', facet='window')
report.figure(fig, treatment_effects, 'rq4_mitigation_paired_effects', "")

In [ ]:
report.table(effect_table(treatment_effects, 'head_click_share'), 'rq4_head_click_effects', "")
report.table(effect_table(treatment_effects, 'head_exposure'), 'rq4_head_exposure_effects', "")
report.table(effect_table(treatment_effects, 'weighted_head_exposure'), 'rq4_weighted_head_exposure_effects', "")

In [ ]:
fig = effect_grid(treatment_effects, ['click_exposure_gap', 'weighted_click_exposure_gap'], title='Click-exposure gap change: mitigation minus control', scales=mitigation_scales, value='value', facet='window')
report.figure(fig, treatment_effects, 'rq4_gap_effects', "")
report.table(effect_table(treatment_effects, 'weighted_click_exposure_gap'), 'rq4_weighted_gap_effects', "")

## Complete trajectories


In [ ]:
fig = trajectory_grid(rq4, ['head_click_share', 'head_exposure', 'weighted_head_exposure'], title='Absolute head shares with and without mitigation', scales=mitigation_scales, facet='model', series='condition_label')
report.figure(fig, rq4, 'rq4_mitigation_trajectories', "")

## In-processing versus post-processing


In [ ]:
method_effects = rq4_effects[rq4_effects['kind'].eq('mitigation_method')]
fig = effect_grid(method_effects, ['head_click_share', 'head_exposure', 'weighted_head_exposure'], title='In-processing minus post-processing', scales=mitigation_scales, value='value', facet='window')
report.figure(fig, method_effects, 'rq4_in_processing_vs_post_processing', "")
report.table(effect_table(method_effects, 'head_click_share'), 'rq4_method_comparison', "")

## Response-model differences versus mitigation changes


In [ ]:
magnitudes = results['mitigation_magnitudes']
for seed in sorted(magnitudes['seed'].unique()):
    selected = magnitudes[magnitudes['seed'].eq(seed)]
    report.table(selected[['model_a', 'model_b', 'focal_model', 'intervention', 'model_absolute_difference_of_means', 'mitigation_absolute_difference_of_means', 'model_mean_absolute_iteration_difference', 'mitigation_mean_absolute_iteration_difference', 'same_feedback_stage']], f'rq4_model_vs_mitigation_seed_{seed}', "")

## Click behaviour


In [ ]:
rq4_summary = summaries[summaries['study'].eq('movielens_mitigation_a300_e804')]
report.table(temporal_table(rq4_summary, ['click_rate', 'no_click_rate', 'valid_clicks']), 'rq4_click_behaviour', "")
fig = trajectory_grid(rq4, ['click_rate', 'no_click_rate'], title='Clicking with and without mitigation', scales=mitigation_scales, facet='model', series='condition_label')
report.figure(fig, rq4, 'rq4_click_behaviour_trajectories', "")

## Catalogue coverage and concentration


In [ ]:
coverage_metrics = ['exposure_coverage', 'click_coverage', 'exposure_gini', 'click_gini']
report.table(temporal_table(rq4_summary, coverage_metrics), 'rq4_coverage_concentration', "")
fig = trajectory_grid(rq4, coverage_metrics, title='Coverage and concentration with and without mitigation', scales=mitigation_scales, facet='model', series='condition_label')
report.figure(fig, rq4, 'rq4_coverage_concentration_trajectories', "")

## Descriptive mean and consistency across available seeds


In [ ]:
from reported_runs.analysis.presentation import cross_seed_table
for metric in ['head_click_share', 'head_exposure', 'weighted_head_exposure']:
    table = cross_seed_table(results, 'rq4', [metric])
    report.table(table, f'rq4_seed_consistency_{metric}', "")